# 03 · Resultados con listas, NumPy y pandas

> **validación humana requerida** antes de usar resultados en ingeniería.

Ejecutamos el voladizo del repo con la wheel local y conservamos IDs, GDL y
unidades al presentar sus desplazamientos. NumPy y pandas son opcionales:
cada importación ausente se indica y el notebook continúa. No se instalan
paquetes ni se calculan respuestas estructurales en Python.
`schemas/engine-input/1.json` (línea 5) y `methods.rs` declaran Experimental solo
`nonlinearStatic`, `soilProfile` y `consolidation1d`; pDelta es Stable; ENG-BENCH-002;
Validated pendiente de ENG-VAL-006. Otras capacidades: validación DESCONOCIDO. Aquí se comprueba la representación del voladizo,
sin validar otras capacidades. [Guía](../docs/GUIA_USUARIO.md).

In [ ]:
from pathlib import Path
import math
import mamut_engine as me

relative = Path("crates/engine-api/fixtures/input/valid/cantilever-linear-static.json")
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents)
             if (p / relative).is_file() and (p / "bindings/python").is_dir()), None)
if root is None:
    raise RuntimeError("Ejecuta dentro del repositorio para localizar el fixture")
model = me.Model.from_json((root / relative).read_text(encoding="utf-8"))
print("Versión y contratos:", me.version())
descriptors = {d["workflow"]["method_id"]: d for d in me.capabilities()["method_descriptors"]}
status = descriptors.get("linearStatic", {}).get("status", "DESCONOCIDO")
print("linearStatic:", "experimental: " + status if status == "Experimental" else status)

In [ ]:
report = me.validate(model, lang="es")
print("Diagnósticos de validación:", report["diagnostics"])
assert not any(d["severity"] == "error" for d in report["diagnostics"])
results = me.run(model, lang="es")
assert results.status in ("ok", "warning") and not results.partial
print("Estado:", results.status, "Diagnósticos:", results.diagnostics)
document = results.to_dict()

## Tablas sin dependencias opcionales

Cada bloque tiene una ruta al caso original. Cada tabla declara sus IDs en
orden de filas, GDL, unidades y ejes; los IDs no son posiciones de fila.
Los resultados nodales mantienen seis columnas aun en D2. Consultamos también
los GDL activos del documento y conservamos diagnósticos y procedencia.

In [ ]:
listed = results.to_arrays()
case_name = model.data["cases"][0]["name"]
case_index = next(i for i, c in enumerate(document["cases"]) if c["case"] == case_name)
block = next(b for b in listed.blocks if b.path == ("cases", case_index))
table = block.matrices["displacements"]
assert table.shape == (len(table.row_ids), len(table.dofs))
assert table.dofs == ("Ux", "Uy", "Uz", "Rx", "Ry", "Rz")
assert table.units == ("m", "m", "m", "rad", "rad", "rad")
assert table.frame == "global"
assert listed.document == document
records = [dict(zip(table.dofs, values), node_id=node_id)
           for node_id, values in zip(table.row_ids, table.values)]
print("Bloque:", block.path, "Shape:", table.shape, "Unidades:", dict(zip(table.dofs, table.units)))
print("GDL activos:", {n["id"]: n["dofs"] for n in document["nodes"]})
print(records)

## NumPy opcional

`to_numpy()` exporta copias float64 en orden C y con memoria propia. Alterar
la copia no altera el snapshot del motor. Si NumPy no se puede importar,
conservamos la tabla de listas de la celda anterior.

In [ ]:
try:
    import numpy as np
except ImportError:
    np = None
    print("NumPy no disponible; se continúa con listas, sin to_numpy().")
else:
    exported = results.to_numpy()
    numpy_block = next(b for b in exported.blocks if b.path == block.path)
    numpy_table = numpy_block.matrices["displacements"]
    values = numpy_table.values
    assert numpy_table.row_ids == table.row_ids
    assert numpy_table.dofs == table.dofs and numpy_table.units == table.units
    assert values.dtype == np.dtype("float64") and values.shape == table.shape
    assert values.flags.c_contiguous and values.flags.owndata and values.base is None
    assert np.array_equal(values, np.asarray(table.values, dtype=np.float64))
    values[0, 0] = 123.0
    assert results.to_dict() == document
    print("NumPy:", np.__version__, "Copia independiente verificada.")

## pandas opcional

Construimos el DataFrame con las listas originales, por lo que la celda no
depende de que se haya ejecutado la rama NumPy. pandas puede requerir NumPy
internamente; si su importación falla también se indica y se continúa.
Las unidades se conservan como metadatos, sin convertir magnitudes.

In [ ]:
try:
    import pandas as pd
except ImportError:
    pd = None
    print("pandas no disponible; se continúa con los registros de listas.")
else:
    frame = pd.DataFrame.from_records(records).set_index("node_id")
    frame.attrs["units"] = dict(zip(table.dofs, table.units))
    frame.attrs["frame"] = table.frame
    assert frame.index.tolist() == list(table.row_ids)
    assert frame.columns.tolist() == list(table.dofs)
    assert frame.values.tolist() == table.values
    print(frame)
    print("Unidades:", frame.attrs["units"], "Ejes:", frame.attrs["frame"])

In [ ]:
assert results.to_dict() == document
print("Estado:", listed.status, "Parcial:", listed.partial)
print("Diagnósticos:", document["diagnostics"])
print("Productor:", results.version, "Huella:", results.input_sha256)
print("Commit:", document.get("git_commit") or "DESCONOCIDO")
print("Presentación completada con las bibliotecas disponibles.")